# 1. Các thư viện cần thiết

## 1.1. Tải thư viện nếu chưa có:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# GHI CHÚ: Pin transformers về bản ổn định 4.44.2 (KHÔNG dùng bản v5.x quá mới vì
# một số model community như ViSoBERT chưa tương thích với cách convert tokenizer
# nội bộ của v5, gây lỗi "argument 'vocab': dict object cannot be converted to Sequence").
# transformers 4.44.2 + tokenizers 0.19.1 là cặp version đã kiểm chứng ổn định với
# PhoBERT, mDeBERTa-v3, CafeBERT và ViSoBERT trong notebook này.
!pip install -q "langdetect"
!pip install -q "pandas" "scikit-learn" "underthesea" "joblib"
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "accelerate>=0.33"
!pip install -q "google-api-python-client"
!pip install -q "wordcloud" "seaborn" "emoji" "openpyxl" "tqdm"

# Sau khi cài xong, BẮT BUỘC Restart session (Runtime > Restart session) rồi chạy lại
# từ đầu, để Colab nạp đúng bản package vừa cài.

## 1.2. Import các thư viện cần thiết:

In [ ]:
# Import thư viện dùng chung:
import pandas as pd  # Thư viện quản lý và xử lý dữ liệu dạng bảng (Đọc file Excel/CSV, lọc dòng, xóa cột...)
import numpy as np   # Thư viện toán học, xử lý ma trận và tính toán số học tốc độ cao.
import re            # Thư viện xử lý biểu thức chính quy (Regex): Dùng để tìm và xóa rác (icon, link, email, ký tự lạ).
import torch

#Các thư viện cho thu thập dữ liệu:
import time          # Thư viện xử lý thời gian: Dùng để tạo độ trễ (sleep) giữa các lần gọi API nhằm tránh bị chặn (Rate Limit).
import json          # Thư viện xử lý JSON: Dùng để đọc và lưu trữ dữ liệu trả về từ API (định dạng chuẩn của dữ liệu mạng xã hội).
import os            # Thư viện hệ điều hành: Dùng để quản lý đường dẫn file, tạo thư mục lưu trữ dữ liệu tự động.
from googleapiclient.discovery import build   # Hàm khởi tạo dịch vụ Google API: Dùng để kết nối và gửi yêu cầu truy xuất dữ liệu tới YouTube Data API v3.
from googleapiclient.errors import HttpError  # Thư viện xử lý lỗi HTTP: Dùng để bắt và xử lý các lỗi kết nối (ví dụ: lỗi 403 hết Quota, lỗi 404 không tìm thấy video).
from datetime import datetime, timedelta, timezone # Thư viện xử lý ngày giờ: Dùng để chuyển đổi timestamp, lọc dữ liệu theo thời gian (ví dụ: chỉ lấy comment trong 1 năm qua).

#Các thư viện cho tiền xử lý DL:
import unicodedata   # Thư viện chuẩn hóa mã Unicode: Giúp sửa lỗi font chữ, đưa tiếng Việt về dạng chuẩn thống nhất.
from langdetect import detect, LangDetectException # Thư viện phát hiện ngôn ngữ - Dùng để phát hiện và lọc bỏ các dòng không phải tiếng Việt.
import collections # Thư viện chứa các cấu trúc dữ liệu hiệu năng cao (như đếm, ngăn xếp).
from sklearn.metrics import confusion_matrix # Hàm tính Ma trận nhầm lẫn: Dùng để đánh giá chi tiết số lượng dự đoán đúng/sai của từng nhãn.
import emoji

#Các thư viện cho trực quan hóa dữ liệu:
import matplotlib.pyplot as plt # Thư viện vẽ biểu đồ
import seaborn as sns # Thư viện trực quan hóa thống kê: Dùng để vẽ biểu đồ đẹp và phức tạp hơn dựa trên Matplotlib.
from wordcloud import WordCloud # Thư viện tạo Mây từ khóa: Dùng để trực quan hóa tần suất xuất hiện của từ trong văn bản
from collections import Counter # Công cụ đếm: Dùng để đếm nhanh số lần xuất hiện của các phần tử trong danh sách
from sklearn.feature_extraction.text import CountVectorizer
from matplotlib.font_manager import FontProperties

#Các thư viện cho việc gán nhãn:
from underthesea import word_tokenize
from transformers import pipeline
from tqdm import tqdm

#Các thư viện cho mô hình hóa:
import joblib        # Dùng để lưu mô hình đã học ra file và tải lại khi cần dùng.(nhẹ hơn pickle)
from sklearn.model_selection import train_test_split # Tách dữ liệu thành 2 phần: Train và test
from sklearn.feature_extraction.text import TfidfVectorizer # Chuyển đổi văn bản thành các con số để máy tính hiểu được.
from sklearn.ensemble import RandomForestClassifier
#5.1. Huấn luyện nhóm Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report # classification_report để soi kỹ
#5.2. Huấn luyện nhóm Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
import gc

print("="*60)
print("1. Đã Import xong thư viện!")
print("="*60)

#Các thư viện còn thiếu (đã bổ sung để sửa lỗi NameError):
from tqdm import tqdm  # Thanh tiến độ khi gán nhãn hàng loạt
from sklearn.model_selection import train_test_split  # Chia tập Train/Test
from sklearn.ensemble import RandomForestClassifier    # Mô hình ML: Random Forest
from sklearn.linear_model import LogisticRegression    # Mô hình ML: Logistic Regression
from sklearn.naive_bayes import MultinomialNB           # Mô hình ML: Naive Bayes
from sklearn.metrics import accuracy_score, f1_score    # Chỉ số đánh giá dùng chung
from torch.utils.data import Dataset                    # Lớp Dataset gốc cho PyTorch
from transformers import Trainer, TrainingArguments     # Bộ huấn luyện HuggingFace Transformers


# 2. Thu thập dữ liệu:

## 2.1. Chiến lược 2 vòng:

In [ ]:
# API Key: đọc từ Colab Secrets (Secrets Manager - icon 🔑 ở sidebar bên trái)
# KHÔNG hardcode key trực tiếp trong code để tránh lộ khóa khi chia sẻ notebook/đẩy lên GitHub.
from google.colab import userdata

API_KEYS = [
    'YT_API_KEY_1',
    'YT_API_KEY_2',
    # Thêm API_Key khác nếu cần: userdata.get('YT_API_KEY_3'),
]
API_KEYS = [k for k in API_KEYS if k]  # loại bỏ key rỗng nếu chưa cấu hình

if not API_KEYS:
    raise ValueError(
        "Chưa cấu hình API Key! Vào Colab Secrets (icon khóa 🔑 bên trái), "
        "thêm secret tên 'YT_API_KEY_1' với giá trị là YouTube Data API Key của bạn."
    )

# Các chủ đề (từ khóa) cần tìm kiếm
DS_Hashtag = [
    "Faker", "world cup 2026", "tìm em", "bão số 1 (maysak)",
    "chuyên tuyên quang", "vtv", "đèo hải vân", "Liên minh huyền thoại","running man"
]

DAYS_BACK = 90          # Chỉ lấy video được đăng trong 90 ngày gần nhất
SCAN_DEPTH = 50         # Mỗi lần tìm kiếm sẽ xem xét 50 video
GLOBAL_TARGET_TOTAL = 12000  # Tổng số comment tối đa muốn thu thập cho toàn bộ dataset


In [ ]:
#================================================================================#
#Vòng 1: Đảm bảo độ phủ (Diversity):
#================================================================================#
# Mỗi từ khóa bắt buộc phải thu thập đủ con số này trước khi sang từ khóa khác.
# Giúp dataset không bị lệch về một chủ đề hot duy nhất.
PHASE_1_CAP_PER_TAG = 1500


#================================================================================#
# VÒNG 2: Tối ưu số lượng (Volume):
#================================================================================#
# Sau khi vòng 1 xong, nếu chưa đủ 12.000 tổng, cho phép các tag "hot".
# Đóng góp thêm tối đa 4.000 comment.
PHASE_2_CAP_PER_TAG = 800

VIDEO_COMMENT_CAP = 300 # Giới hạn: 1 video chỉ lấy tối đa 300 comment (tránh 1 video viral chiếm hết dữ liệu)
DATA_FILE = "data_moi.csv"      # Tên file chứa dữ liệu kết quả
STATE_FILE = "crawler_2phases_state.json" # Tên file lưu trạng thái (để khi chạy lại không bị trùng lặp)

# Từ khóa để lọc bỏ các video không phù hợp (nhạc không lời, karaoke,...)
Keyword_Kophuhop = ['official mv', 'music video', 'bản gốc', 'lyrics',
                    'lời bài hát', 'karaoke', 'vietsub', '1 hour', 'remix only']

## 2.2. Lớp quản lý API (API Handler Class):

In [ ]:
# Khai báo lớp quản lý kết nối YouTube API:
class YouTubeAPIHandler:
    #==============================================================================================================#
    # Hàm khởi tạo (Constructor): Chạy đầu tiên khi gọi YouTubeAPIHandler(API_KEYS)
    #==============================================================================================================#
    def __init__(self, keys):
        self.keys = keys # Lưu trữ danh sách các API Key bạn cung cấp vào biến của lớp
        self.current_index = 0 # Thiết lập chỉ số bắt đầu là 0 (tức là dùng Key đầu tiên trong danh sách)
        self.service = self._build_service() # Khởi tạo kết nối đầu tiên

    #==============================================================================================================#
    # Hàm nội bộ (bắt đầu bằng _) để tạo đối tượng service
    #==============================================================================================================#
    def _build_service(self):
        """Tạo đối tượng kết nối với YouTube dùng key hiện tại"""
        # Kiểm tra an toàn: Nếu chỉ số hiện tại (current_index) lớn hơn hoặc bằng tổng số key
        if self.current_index >= len(self.keys):
            # Nếu đã dùng hết danh sách Key mà vẫn lỗi -> Dừng chương trình
            raise Exception("TẤT CẢ API KEY ĐÃ HẾT HẠN MỨC! Vui lòng thêm key mới.")

        print(f"Đang kích hoạt API Key số {self.current_index + 1}...")
        return build('youtube', 'v3', developerKey=self.keys[self.current_index])

    #==============================================================================================================#
    # Hàm xử lý khi gặp lỗi (thường là lỗi 403 Quota Exceeded)
    #==============================================================================================================#
    def _rotate_key(self):
        """Cơ chế tự động đổi chìa khóa (Switching)"""
        print(f"Key số {self.current_index + 1} gặp lỗi hoặc hết quota. Đang đổi key...")
        self.current_index += 1 # Tăng index để lấy key tiếp theo
        self.service = self._build_service() # Tái khởi tạo dịch vụ

    #==============================================================================================================#
    # Hàm thực thi yêu cầu (Wrapper Function):
    #==============================================================================================================#
    def execute_request(self, function_name, **kwargs):
        """
        Hàm bao bọc (Wrapper) thông minh:
        - Nhận yêu cầu (tìm kiếm, lấy video, lấy comment).
        - Thực thi trong vòng lặp 'while True' để retry nếu cần.
        """
        while True:
            try:
                # Chọn hàm API tương ứng dựa trên tham số function_name
                if function_name == 'search':
                    return self.service.search().list(**kwargs).execute()
                elif function_name == 'videos':
                    return self.service.videos().list(**kwargs).execute()
                elif function_name == 'comments':
                    return self.service.commentThreads().list(**kwargs).execute()
            except HttpError as e:
                # Lỗi 1: "Video tắt bình luận"
                # Đây là lỗi nội dung, không phải lỗi hệ thống -> Bỏ qua video này, không đổi key.
                if e.resp.status == 403 and b'commentsDisabled' in e.content:
                    print(f"Video tắt tính năng bình luận. Bỏ qua.")
                    return None

                # Lỗi 2: Hết hạn mức (Quota Exceeded - 403) hoặc quá nhiều yêu cầu (429) -> Đổi sang API Key dự phòng.
                if e.resp.status in [403, 429]:
                    print(f"Lỗi Quota / Hết hạn mức: {e}")
                    self._rotate_key() # Gọi hàm đổi key
                    continue # Quay lại đầu vòng while để thử lại với key mới
                # Các lỗi khác (VD: 404 Not Found) -> Bỏ qua
                else:

                    print(f"Lỗi API khác (Ignored): {e}")
                    return None
            except Exception as e:
                print(f"Lỗi không xác định: {e}")
                return None

## 2.3. Các hàm bổ trợ (Helper Functions):

In [ ]:
#==============================================================================================================================#
# Hàm giúp khôi phục tiến độ làm việc nếu code bị ngắt giữa chừng. Đọc file JSON chứa danh sách vidoe đã quét -> Tránh trùng lặp.
#==============================================================================================================================#
def load_state():
    # Kiểm tra xem file lưu trạng thái có tồn tại không
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r', encoding='utf-8') as f:
            state = json.load(f)
            state['processed_videos'] = set(state['processed_videos']) # Chuyển list thành set để tra cứu nhanh hơn
            return state
    # Nếu file chưa tồn tại (lần chạy đầu tiên), trả về trạng thái mặc định rỗng
    return {'processed_videos': set(), 'hashtag_counts': {}, 'total_collected': 0}

#==============================================================================================================================#
# Lưu lại tiến độ hiện tại vào file JSON, hàm này được gọi liên tục sau mỗi lần quét xong 1 lô video.
#==============================================================================================================================#
def save_state(state):
    state_to_save = state.copy() # Tạo bản sao để không ảnh hưởng dữ liệu đang chạy
    state_to_save['processed_videos'] = list(state['processed_videos']) # JSON không lưu được set, phải chuyển về list
    # Ghi đè vào file cũ với định dạng UTF-8 để không lỗi font
    with open(STATE_FILE, 'w', encoding='utf-8') as f:
        json.dump(state_to_save, f, ensure_ascii=False, indent=2)

#==============================================================================================================================#
# Lưu DL ngay lập tức mỗi khi có DL mới -> Tránh mất sạch DL nếu code bị crash
#==============================================================================================================================#
def save_data_incremental(data_list):
    if not data_list: return # Nếu không có dữ liệu thì thoát
    df = pd.DataFrame(data_list)
    # Nếu file chưa tồn tại thì ghi tiêu đề (header), nếu có rồi thì chỉ ghi nối tiếp dữ liệu
    header = not os.path.exists(DATA_FILE)
    df.to_csv(DATA_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')

#==============================================================================================================================#
# Loại bỏ các video không phù hợp để phân tích
#==============================================================================================================================#
def is_passive_content(title):
    for kw in Keyword_Kophuhop:
        if kw in title.lower(): return True
    return False

#==============================================================================================================================#
# Chuyển thời gian thành tổng số giây. Vì Youtube trả về định dạng ISO 8601 (VD: PT1H2M - 1 giờ 2 phút).
#==============================================================================================================================#
def parse_duration(duration_iso):
    match = re.match(r'PT(\d+H)?(\d+M)?(\d+S)?', duration_iso)
    if not match: return 0
    h = int(match.group(1)[:-1]) if match.group(1) else 0
    m = int(match.group(2)[:-1]) if match.group(2) else 0
    s = int(match.group(3)[:-1]) if match.group(3) else 0
    return h*3600 + m*60 + s

## 2.4. Logic thu thập dữ liệu:

In [ ]:
#=======================================================================================================#
# Hàm xử lý chính cho 1 từ khóa:
# 1. Tìm video theo từ khóa.
# 2. Lọc vidoe rác và lấy thông tin.
# 3. Cào bình luận.
# 4. Lưu DL.
#=======================================================================================================#
def process_hashtag(api_handler, tag, target_cap, state, published_after):
    current_tag_count = state['hashtag_counts'].get(tag, 0) # Lấy số lượng comment hiện có của tag này từ biến 'state' (bộ nhớ tạm)

    # Nếu từ khóa này đã đủ chỉ tiêu (VD: đã có 1000 cmt ở vòng 1) -> Bỏ qua
    if current_tag_count >= target_cap:
        print(f"Tag '{tag}' đã đủ chỉ tiêu vòng này ({current_tag_count}/{target_cap}). Skip.")
        return

    print(f"\nQuét: '{tag}' (Đang có: {current_tag_count} | Mục tiêu vòng này: {target_cap})")

    #=======================================================================================================#
    #Bước 1: Tìm Video (Dùng Handler):
    #=======================================================================================================#

    # Gọi API Search để lấy danh sách Video ID liên quan đến từ khóa
    search_res = api_handler.execute_request(
        'search',
        q=tag, part='id', maxResults=SCAN_DEPTH, type='video',
        order='relevance', publishedAfter=published_after, # Lọc theo ngày
        regionCode='VN', relevanceLanguage='vi'            # Ưu tiên nội dung Việt Nam
    )
    # Nếu API không trả về kết quả hoặc bị lỗi -> Dừng hàm
    if not search_res: return

    # Lấy danh sách ID video, loại bỏ các video đã từng quét (nằm trong state['processed_videos'])
    video_ids_raw = [item['id']['videoId'] for item in search_res.get('items', [])]
    # Chỉ giữ lại những video chưa từng được xử lý (không nằm trong processed_videos)
    video_ids = [vid for vid in video_ids_raw if vid not in state['processed_videos']]

    if not video_ids:
        print("Không còn video mới cho tag này.")
        return

    #=======================================================================================================#
    #Bước 2: Lấy chi tiết Video và lọc (Videos API):
    #=======================================================================================================#
    # YouTube API cho phép lấy thông tin tối đa 50 video một lần để tiết kiệm quota.

    candidates = [] # Danh sách video tiềm năng sẽ được cào comment.
    for i in range(0, len(video_ids), 50):
        chunk = video_ids[i:i+50]
        # Gọi API 'videos' để lấy chỉ số thống kê (view, like, comment count) và chi tiết (duration).
        stats_res = api_handler.execute_request(
            'videos', part='snippet,statistics,contentDetails', id=','.join(chunk)
        )
        if not stats_res: continue

        for item in stats_res.get('items', []):
            # Lọc nội dung thụ động (Karaoke, MV gốc...)
            if is_passive_content(item['snippet']['title']): continue

            # Phân loại Short/Long video dựa trên độ dài (<= 60s là Short)
            sec = parse_duration(item['contentDetails']['duration'])
            candidates.append({
                'id': item['id'],
                'title': item['snippet']['title'],
                'type': "SHORT" if sec <= 60 else "LONG",
                'channel': item['snippet']['channelTitle'],
                'published_at': item['snippet']['publishedAt'],
                'views': int(item['statistics'].get('viewCount', 0)),
                'comment_count': int(item['statistics'].get('commentCount', 0)),
                'likes': int(item['statistics'].get('likeCount', 0))
            })

    # Sắp xếp video theo lượt xem giảm dần (Ưu tiên lấy video hot trước)
    candidates.sort(key=lambda x: x['views'], reverse=True)

    #=======================================================================================================#
    #Bước 3: Lấy bình luận (CommentThreads API):
    #=======================================================================================================#
    batch_data = [] # Chứa các comment thu thập được trong đợt này

    for vid in candidates:
        # Kiểm tra các điều kiện dừng (Đủ target toàn cục hoặc đủ target cho tag này)
        if state['total_collected'] >= GLOBAL_TARGET_TOTAL: break # Nếu tổng kho dữ liệu đã đủ 12.000 dòng -> Dừng toàn bộ
        if current_tag_count >= target_cap: break # Nếu từ khóa này đã đủ chỉ tiêu -> Dừng xử lý từ khóa này.

        # Tính toán số lượng comment cần lấy từ video này
        remaining_tag = target_cap - current_tag_count # Còn thiếu bao nhiêu cho tag này
        remaining_global = GLOBAL_TARGET_TOTAL - state['total_collected'] # Còn thiếu bao nhiêu cho tổng

        # Lấy min để không bao giờ vượt quá giới hạn (VD: Chỉ lấy tối đa 300 cmt/video)
        to_take = min(vid['comment_count'], VIDEO_COMMENT_CAP, remaining_tag, remaining_global)

        # Nếu không cần lấy thêm (to_take <= 0), đánh dấu đã xử lý và bỏ qua
        if to_take <= 0:
            state['processed_videos'].add(vid['id'])
            continue

        # Gọi API lấy comment
        cmt_res = api_handler.execute_request(
            'comments',
            part='snippet', videoId=vid['id'], maxResults=to_take,
            textFormat='plainText', order='relevance' # Lấy comment phù hợp nhất
        )

        if cmt_res:
            temp_cmts = []
            for t in cmt_res.get('items', []):
                # Trích xuất nội dung comment và thông tin người dùng
                c = t['snippet']['topLevelComment']['snippet']
                temp_cmts.append({
                    'Hashtag': tag, 'Video Type': vid['type'],
                    'Video Title': vid['title'], 'Views': vid['views'],
                    'Comment Text': c['textDisplay'],
                    'Comment Likes': int(c['likeCount']),
                    'Comment Date': c['publishedAt']
                })

            if temp_cmts:
                # Sắp xếp comment theo like (lấy comment chất lượng nhất)
                temp_cmts.sort(key=lambda x: x['Comment Likes'], reverse=True)
                batch_data.extend(temp_cmts)

                # Cập nhật các bộ đếm
                fetched = len(temp_cmts)
                current_tag_count += fetched
                state['total_collected'] += fetched
                state['hashtag_counts'][tag] = current_tag_count
                print(f"      + Lấy {fetched} cmt từ: {vid['title'][:20]}...")

        # Đánh dấu video này đã xử lý xong
        state['processed_videos'].add(vid['id'])

    #=======================================================================================================#
    #Bước 4: # Lưu dữ liệu xuống ổ cứng sau khi xử lý xong một batch video:
    #=======================================================================================================#
    save_data_incremental(batch_data)
    save_state(state)

## 2.5. Chạy thu thập DL:

In [ ]:
def run_two_phase_strategy():

    # Kiểm tra xem người dùng đã điền Key chưa
    if len(API_KEYS) < 1:
        print("CẢNH BÁO: Bạn chưa nhập API Key!!!")
        return
    # Khởi tạo trình quản lý API (để xử lý việc đổi key tự động)
    try:
        api_handler = YouTubeAPIHandler(API_KEYS)
    except Exception as e:
        print(e)
        return

    state = load_state() # Khôi phục trạng thái cũ (nếu có)

    # Tính mốc thời gian (VD: Lấy từ 90 ngày trước đến nay)
    now = datetime.now(timezone.utc)
    start_date = now - timedelta(days=DAYS_BACK)
    published_after = start_date.isoformat().replace('+00:00', 'Z') # Format chuẩn cho YouTube API

    print("="*60)
    print("BẮT ĐẦU CHIẾN DỊCH 2 VÒNG (AUTO KEY ROTATION & FIX LỖI 403)")
    print("="*60)

    #================================================================================#
    #Vòng 1: Chạy đầu:
    #================================================================================#
    # Mục tiêu: Mỗi hashtag phải kiếm đủ 1000 comment.
    # Nếu tag nào ít video quá thì chịu, nhưng tag hot cũng chỉ lấy 1000 rồi dừng.
    print(f"\n🏁 [VÒNG 1] ĐẢM BẢO MỖI TAG CÓ {PHASE_1_CAP_PER_TAG} COMMENT")
    for tag in DS_Hashtag:
        if state['total_collected'] >= GLOBAL_TARGET_TOTAL: break
        process_hashtag(api_handler, tag, PHASE_1_CAP_PER_TAG, state, published_after)

    #================================================================================#
    #Vòng 2: Tăng tốc (Vét):
    #================================================================================#
    # Sau vòng 1, nếu tổng dataset chưa đủ 12.000 dòng,
    # quay lại các tag hot và cho phép lấy thêm tới 4.000 comment/tag để bù vào.
    if state['total_collected'] < GLOBAL_TARGET_TOTAL:
        print(f"\n🏁 [VÒNG 2] TĂNG TỐC VỚI TAG HOT (LIMIT {PHASE_2_CAP_PER_TAG})")
        for tag in DS_Hashtag:
            if state['total_collected'] >= GLOBAL_TARGET_TOTAL: break
            process_hashtag(api_handler, tag, PHASE_2_CAP_PER_TAG, state, published_after)

    print(f"\nHOÀN TẤT! Tổng thu thập: {state['total_collected']}/{GLOBAL_TARGET_TOTAL}")

if __name__ == "__main__":
    # Kiểm tra an toàn: Người dùng phải thay thế chuỗi mặc định
    if API_KEYS[0] == 'YOUR_API_KEY_1':
        print("VUI LÒNG ĐIỀN ÍT NHẤT 1 API KEY VÀO LIST!")
    else:
        run_two_phase_strategy()

# 3. Tiền xử lý DL:

## 3.1. Tạo danh mục các từ viết tắt dựa trên file dữ liệu

### 3.1.1. Đọc file DMviettat_Goiy.csv từ ViLexNorm:

In [ ]:
try:
    #Đọc dữ liệu từ file
    df_dev = pd.read_csv('DMviettat_Goiy.csv')
    #Chỉ lấy các cột cần thiết
    df_dev = df_dev[['original', 'normalized']]
except Exception as e:
    print("Lỗi đọc file DMviettat_Goiy.csv:", e)
    df_dev = pd.DataFrame() #Tạo rỗng nếu lỗi

#Tạo danh sách từ điển viết tắt
DS_Teencode = {}
print("="*60)
print("Đọc file: Xong")
print("="*60)
print("Tạo danh mục viết tắt: Xong")
print("="*60)

### 3.1.2. Học từ điển tự động từ dữ liệu có sẵn:

In [ ]:
#Logic: So sánh từng từ trong câu gốc và câu chuẩn. Nếu khác nhau -> Là từ viết tắt.
for index, row in df_dev.iterrows():
    try:
        tu_VietTat = str(row['original']).lower().split()
        tu_Dung = str(row['normalized']).lower().split()

        #Chỉ học từ những câu có cấu trúc tương đồng (số từ bằng nhau) để tránh lỗi
        if len(tu_VietTat) == len(tu_Dung):
            for o, n in zip(tu_VietTat, tu_Dung):
                if o != n and o not in DS_Teencode:
                    DS_Teencode[o] = n
    except Exception as e:
        print("Lỗi:", e)

print("="*60)
print("Lưu các từ viết tắt vào danh sách các từ viết tắt: Xong")
print("="*60)

### 3.1.3. Bổ sung Từ điển về các trend và mạng xã hội (Để phù hợp với dữ liệu của mình):

In [ ]:
DS_VietTat_BoSung = {
    # Mạng xã hội chung
    "fl": "theo dõi", "sub": "đăng ký", "view": "lượt xem", "share": "chia sẻ",
    "cmt": "bình luận", "ib": "nhắn tin", "ad": "quản trị viên", "tus": "trạng thái",
    "rep": "trả lời", "viral": "lan truyền", "drama": "bê bối", "acc": "tài khoản",
    "avt": "ảnh đại diện", "cover": "ảnh bìa", "mem": "thành viên", "gr": "nhóm",
    "clip": "video", "fan": "người hâm mộ", "idol": "thần tượng",
    "bão": "nhiều", "tym": "tim", "like": "thích","mắc cừ": "cười", "tloi": "trả lời", "pv": "phỏng vấn",
    "respect": "tôn trọng", "vid": "video",

    # Teencode phổ biến (Bổ trợ thêm cho danh sách ViLexNorm)
    "ko": "không", "hok": "không", "k": "không", "kh": "không", "hong": "không",
    "dc": "được", "đc": "được", "dk": "được",
    "j": "gì", "ji": "gì",
    "wa": "quá", "wá": "quá", "qa": "quá",
    "iu": "yêu", "yeu": "yêu", "thik": "thích",
    "vn": "việt nam", "nt": "nhắn tin",
    "uk": "ừ", "uhm": "ừ", "r": "rồi",
    "huhu": "khóc", "kkk": "cười", "hihi": "cười",
    "ng": "người", "nng": "người", "ngta": "người ta",
    "m": "mày", "t": "tao", "mik": "mình", "mk": "mình",
    "ck": "chồng", "vk": "vợ", "cr": "người yêu đơn phương", "ny": "người yêu", "cs": "có","chs": "chơi",

    #Gaming / Esports
    "gank": "tấn công", "mid": "đường giữa", "top": "đường trên", "ad": "xạ thủ",
    "sp": "hỗ trợ", "win": "thắng", "lose": "thua", "game": "trò chơi",
    "pro": "hay", "noob": "dở", "ga": "dở", "lag": "giật",
    "combat": "giao tranh", "farm": "kiếm tiền", "cktg": "chung kết thế giới",

    #Từ lóng & Drama (Slang)
    "7 học": "thất học", "ấm dâu": "ấu dâm", "pbvm": "phân biệt vùng miền",
    "qrtd": "quấy rối tình dục", "drm": "drama/phốt", "sếch joke": "đùa tình dục",
    "sv": "sinh viên", "bvs": "băng vệ sinh", "kols": "người có sức ảnh hưởng",
    "antifan": "người tẩy chay", "fandom": "cộng đồng fan", "comeback": "trở lại",
    "viral": "lan truyền mạnh", "top trending": "thịnh hành", "bias": "thần tượng yêu thích nhất",

    #Tên riêng & Chương trình (Showbiz)
    "atsh": "anh trai say hi", "htrr": "hành trình rực rỡ", "exsh": "em xinh say hi",
    "hth": "hieuthuhai", "pmc": "phương mỹ chi", "tt": "trấn thành",
    "ml": "miu lê", "qh": "quang hùng", "qhmtd": "quang hùng masterd",
    "st": "sơn tùng", "mtp": "sơn tùng m-tp", "vct": "vũ cát tường",
    "acn": "anh công nghệ", "a05": "cục an ninh mạng", "vieon": "ứng dụng vieon"
}

# Cập nhật thêm từ vào từ điển gốc
DS_Teencode.update(DS_VietTat_BoSung)
print("="*60)
print(f"Đã xây dựng xong bộ từ điển gồm {len(DS_Teencode)} từ viết tắt.")
print("="*60)

In [ ]:
#Lưu từ điển
df_Teencode = pd.DataFrame(list(DS_Teencode.items()), columns=['Từ viết tắt', 'Từ đúng'])
df_Teencode.to_csv('DS_Teencode.csv', index=False, encoding='utf-8-sig')

# Xem 10 dòng đầu tiên
print("="*60)
print("Danh sách từ điển teencode")
print("="*60)
print(df_Teencode.head(10))
print("="*60)

## 3.2. Tiền xử lý DL:

### Chuyển file csv thành file excel:

In [ ]:
def convert_csv_to_excel(csv_file, excel_file):
    print("="*60)
    print("Chuyển đổi file csv -> Excel...")
    print("="*60)
    if not os.path.exists(csv_file):
        print(f"Lỗi: Không tìm thấy file CSV '{csv_file}' để chuyển đổi.")
        return

    try:
        # Đọc file CSV (xử lý encoding utf-8 để không lỗi font tiếng Việt)
        df_DLtho = pd.read_csv(csv_file, encoding='utf-8-sig')

        # 2. Lưu sang Excel (index=False để bỏ cột số thứ tự thừa)
        df_DLtho.to_excel(excel_file, index=False, engine='openpyxl')

        print("Thành công! File Excel đã được tạo")
        print("Tổng số dòng dữ liệu:",len(df_DLtho))

    except Exception as e:
        print("Có lỗi xảy ra khi chuyển đổi:", e)

In [ ]:
# Cấu hình các file (File cần chuyển và tên file xuất ra)
csv_name = "data_moi.csv"      # Tên file CSV đầu vào (kết quả từ code cào)
excel_name = "data_moi.xlsx"     # Tên file Excel đầu ra (để bạn dùng cho Tiền xử lý)
# Chạy hàm chuyển file csv -> excel
convert_csv_to_excel(csv_name, excel_name)

### 3.2.1. Đọc dữ liệu:

In [ ]:
file_path = 'data_moi.xlsx' #Lưu đường dẫn file
print("="*60)
try:
    df = pd.read_excel(file_path, engine='openpyxl') #Đọc dữ liệu
    print("Đã đọc dữ liệu thành công!")
    print("="*60)
    print("Kích thước bộ dữ liệu gốc:",df.shape)
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại đường dẫn '{file_path}'. Hãy kiểm tra lại tên file hoặc đường dẫn.")
except ImportError:
    print("Lỗi: Thiếu thư viện 'openpyxl'. Hãy chạy lệnh: pip install openpyxl")
except Exception as e:
    print("Lỗi đọc file:", e)

### 3.2.2. Kiểm tra dữ liệu trùng lặp:

In [ ]:
print("="*60)
print("Xử lý DL trùng lặp: ")
print("="*60)
slTrungLap = df.duplicated().sum()
print("Số lượng dòng trùng lặp: ",slTrungLap)
print("="*60)
df = df.drop_duplicates() #Xóa dòng trùng
print("Đã xóa các dòng trùng ")
print("="*60)

### 3.2.3. Xử lý giá trị bị thiếu:

In [ ]:
print("="*60)
print("Xử lý giá trị bị thiếu: ")
print("="*60)
print("Danh sách số lượng các cột có dòng bị thiếu: ")
print(df.isnull().sum())
#Xóa dòng thiếu comment
df = df.dropna(subset=['Comment Text'])
print("="*60)
print("Đã xóa các dòng bị thiếu")
print("="*60)

### 3.2.4. Tách icon và văn bản:

In [ ]:
#Định nghĩa lại biến Regex toàn cục để dùng chung
VN_ChuCai = r'a-zA-Z0-9àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴÈÉẸẺẼÊỀẾỆỂỄÌÍỊỈĨÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠÙÚỤỦŨƯỪỨỰỬỮỲÝỴỶỸĐ'


print("="*60)
print("Xử lý tách Icon và Văn bản")
print("="*60)
def strict_separation_handler(text):
    #Xử lý trường hợp không phải chuỗi (NaN, float...)
    if not isinstance(text, str):
        return pd.Series(["", ""], index=['Comment Text', 'Comment Icon'])

    #Pattern: Tìm những gì không phải là (Chữ cái + Số + Khoảng trắng)
    #f-string sẽ tạo ra chuỗi: [^a-zA-Z... \s]
    pattern_non_text = f'[^{VN_ChuCai}\s]'

    #1.Tạo comment text (Chỉ giữ lại Chữ + Số + Khoảng trắng)
    clean_text = re.sub(pattern_non_text, ' ', text) #Thay thế tất cả các ký tự lạ (Icon, dấu câu) bằng khoảng trắng
    clean_text = re.sub(r'\s+', ' ', clean_text).strip() #Xóa khoảng trắng thừa

    #2. Tạo comment icon (Lấy phần còn lại)
    #Tìm tất cả các ký tự lạ
    pattern_get_icon = f'[^{VN_ChuCai}\s]'#Thêm \s vào pattern để KHÔNG lấy khoảng trắng vào cột icon
    icons_list = re.findall(pattern_get_icon, text)
    icons_str = "".join(icons_list)

    return pd.Series([clean_text, icons_str], index=['Comment Text', 'Comment Icon'])

In [ ]:
#Áp dụng hàm vào dữ liệu của mình (df)
try:
    #Nếu file gốc chỉ có 'Comment Text', sao chép nó ra cột 'Comment' để lưu trữ
    if 'Comment' not in df.columns:
        if 'Comment Text' in df.columns:
            df['Comment'] = df['Comment Text']

    df['Comment'] = df['Comment'].fillna("")

    #Thực thi tách dữ liệu (Ghi đè hoặc tạo mới cột)
    df[['Comment Text', 'Comment Icon']] = df['Comment'].apply(strict_separation_handler)

    print("="*60)
    print("Tách icon và văn bản: XONG!")
    print("="*60)
    print("Kiểm tra nhanh 3 dòng đầu sau khi tách:")
    print(df[['Comment', 'Comment Text', 'Comment Icon']].head(3))

    print("="*60)
    #Tiền xử lý dữ liệu sau khi tách
    print("Loại bỏ các dòng rỗng")
    rows_before = len(df)

    #Điều kiện: Text rỗng và icon cũng rỗng
    is_text_empty = df['Comment Text'].str.strip() == ''
    is_icon_empty = df['Comment Icon'].str.strip() == ''
    mask_empty = is_text_empty & is_icon_empty

    #Xóa các dòng có không có comment dạng chữ và icon
    df = df[~mask_empty].copy()
    print(f"-> Đã xóa {rows_before - len(df)} dòng rỗng (Không text + Không icon).")
    print("-> Số dòng còn lại:", len(df))
    print("="*60)

except Exception as e:
    print("Lỗi:", e)

### 3.2.5. Chuẩn hóa dữ liệu:

In [ ]:
print("="*60)
print("Hàm chuẩn hóa dữ liệu (Teencode)")
print("="*60)

#Hàm chuẩn hóa DL:
def normalize_comment_advanced(text):

    #1. KT DL vào:
    if not isinstance(text, str): return "" # Nếu DL bị lỗi (Không phải chuỗi) -> Trả về rỗng

    #2. Đưa toàn bộ văn bản về bảng mã chuẩn
    text = unicodedata.normalize('NFC', text)

    #3. Tách từ
    words = text.split() # Cắt câu thành danh sách các từ dựa vào khoảng trắng
    corrected_words = [] # Tạo một danh sách rỗng để chứa các từ sau khi sửa

    #4. Duyệt từng từ để sửa lỗi
    for word in words:
        clean_word_key = word.lower() #Chuyển thành chữ thường

        #5. Tra từ điển viết tắt
        if clean_word_key in DS_Teencode:
            corrected_words.append(DS_Teencode[clean_word_key]) #Nếu là từ viết tắt, lấy từ chuẩn trong từ điển bỏ vào danh sách kết quả.
        else:
            corrected_words.append(word) #Nếu ko phải từ viết tắt, giữ nguyên từ gốc bỏ vào danh sách

    #6. Nối các từ đã sửa thành câu
    return " ".join(corrected_words)

In [ ]:
#Áp dụng chuẩn hóa lên cột 'Comment Text' (Cột đã sạch icon)
try:
    print("="*60)
    df['Comment Text'] = df['Comment Text'].apply(normalize_comment_advanced)
    print("Chuẩn hóa dữ liệu: XONG!")
    print("="*60)
except Exception as e:
    print("Lỗi:", e)

### 3.2.6. Lọc bỏ các dòng chứa ngôn ngữ nước ngoài:

In [ ]:
try:
    print("="*60)
    print("Lọc bỏ các dòng chứa ngôn ngữ nước ngoài trong cột 'Comment Text'")
    print("="*60)

    def smart_language_filter(text):
        #1. Bảo vệ các câu quá ngắn (Nhỏ hơn 15 ký tự và các câu như 'ok','t1 win',..):
        if not isinstance(text, str) or len(text.strip()) <= 15:
            return True

        #2. Bảo vệ các câu có từ tiếng Việt:
        #VD: "Marisa đã xem" -> Có chữ "đ", "ã" -> Giữ lại
        if re.search(VN_ChuCai, text.lower()):
            return True

        #3. Dùng LANGDETECT để bắt các câu dài không có dấu:
        try:
            lang = detect(text)

            #Nếu máy đoán là Tiếng Việt ('vi') -> Giữ
            if lang == 'vi':
                return True

            #Nếu máy đoán là tiếng Anh, Trung, Hàn, Nhật,... -> Xóa (Chỉ xóa khi KHÔNG có dấu tiếng Việt ở Bước 2)
            if lang in ['en', 'zh-cn', 'ko', 'ja', 'ru']:
                return False

            #Các trường hợp còn lại (ngôn ngữ lạ) -> Giữ lại cho an toàn
            return True

        except LangDetectException:
            return True

    #Đếm số dòng trước khi lọc
    rows_before = len(df)

    #Áp dụng bộ lọc mới
    is_vietnamese = df['Comment Text'].apply(smart_language_filter)

    #4. Xem thử những dòng sẽ bị xóa (Để kiểm tra xem còn xóa nhầm không)
    removed_rows = df[~is_vietnamese]
    if not removed_rows.empty:
        print("Các dòng sẽ bị xóa:")
        # In ra cả câu để đánh giá
        print(removed_rows['Comment Text'].sample(min(5, len(removed_rows))).values)

    #5. Thực hiện xóa
    df = df[is_vietnamese].copy()

    print("="*60)
    print("Đã xử lý xong.")
    print("- Số dòng ban đầu:", rows_before)
    print("- Số dòng bị loại bỏ (Tiếng Anh/Hàn/Trung...):", rows_before - len(df))
    print("- Số dòng còn lại:", len(df))
except Exception as e:
    print("Lỗi:", e)

### 3.2.7. Lọc bỏ StopWords:

In [ ]:
STOPWORDS = {
    'là', 'của', 'và', 'những', 'các', 'thì', 'mà', 'cái', 'việc', 'bộ',
    'để', 'này', 'kia', 'trong', 'trên', 'dưới', 'gì', 'khi', 'bị', 'bởi',
    'cả', 'như', 'vậy', 'rằng', 'sẽ', 'đang', 'đã', 'nên', 'với', 'tại',
    'qua', 'đến', 'từ', 'có', 'người', 'ra', 'vào', 'lại', 'cái', 'được', 'cho', 'đó','tôi','mình','rồi','cũng',
    'chỉ','ơi','á','à','ừ','nhỉ','luôn','đi','nữa','chứ','thôi','thế','chúng', 'rất', 'nha', 'nhé','quá','vẫn','còn'
}

# Hàm loại bỏ Stopwords
def remove_stopwords(text):
    words = text.split() # Tách từ
    # Chỉ giữ lại các từ KHÔNG nằm trong danh sách STOPWORDS
    clean_words = [word for word in words if word not in STOPWORDS]
    return ' '.join(clean_words) # Ghép lại thành câu


print("="*60)
print("Lọc bỏ StopWords")

col_name = 'Comment Text'

if col_name in df.columns:
    df['Comment Text'] = df['Comment Text'].apply(remove_stopwords)
    print("="*60)
    print("Đã lọc bỏ StopWords! Dữ liệu mẫu:")
    print(df[col_name].head())
    print("="*60)
else:
    print("="*60)
    print(f"Lỗi: Không tìm thấy cột '{col_name}' trong dữ liệu.")
    print("="*60)

## 3.3. Xử lý icon:

### 3.3.1. Xử lý ngôn ngữ khác trong cột Comment Icon:

In [ ]:
print("="*60)
print("Loại bỏ ngôn ngữ khác trong cột 'Comment Icon'")
print("="*60)

# 1. Hàm lọc chỉ giữ lại Emoji
def keep_only_emojis(text):
    # Nếu ô rỗng hoặc không phải chuỗi -> Trả về rỗng
    if not isinstance(text, str):
        return ""

    # Duyệt qua từng ký tự, chỉ giữ lại cái nào là Emoji
    # emoji.is_emoji(char) trả về True nếu ký tự là icon
    cleaned_text = "".join([char for char in text if emoji.is_emoji(char)])

    return cleaned_text

# 2. Áp dụng vào cột Comment Icon
try:
    # Kiểm tra trước khi xử lý (Demo vài dòng có tiếng nước ngoài nếu có)
    # Ví dụ lọc thử các dòng có chứa ký tự không phải ASCII để xem
    print("Ví dụ trước khi xử lý:")
    print(df['Comment Icon'].head())

    # Thực hiện làm sạch
    df['Comment Icon'] = df['Comment Icon'].apply(keep_only_emojis)

    print("="*60)
    print("Ví dụ sau khi xử lý (Chỉ còn icon):")
    print(df['Comment Icon'].head())

    print("="*60)
    print("Đã loại bỏ hoàn toàn ngôn ngữ khỏi cột Icon!")
    print("="*60)

except Exception as e:
    print("Lỗi:",e)
    print("="*60)

### 3.3.2. Thu thập tất cả các Icon:

In [ ]:
print("="*60)
print("Thu thập các icons...")

icons_duynhat = set() #Dùng set để tự động loại bỏ trùng lặp
ds_all_icons = []  #Dùng list để đếm tần suất (nếu cần)

if 'Comment Icon' in df.columns:
    #Chuyển cột về dạng chuỗi và xử lý NaN
    icon_series = df['Comment Icon'].fillna("").astype(str)

    print("="*60)
    print("Quét và tách các icon duy nhất")
    for text in icon_series:
        #Duyệt qua từng ký tự trong chuỗi icon
        for char in text:
            #Bỏ qua khoảng trắng
            if char.strip() != "":
                icons_duynhat.add(char)
                ds_all_icons.append(char)

    print("Số loại icon khác nhau:",len(icons_duynhat))
else:
    print("Không tìm thấy cột 'Comment Icon'. Hãy kiểm tra lại tên cột.")

### 3.3.3. Tạo dataframe và sắp xếp theo thứ tự icon xuất hiện nhiều nhất:

In [ ]:
icon_SL = collections.Counter(ds_all_icons)

#Tạo bảng dữ liệu: [Icon, Số lần xuất hiện, Nghĩa tiếng Việt (để trống)]
bang_Icons = []
for icon in icons_duynhat:
    bang_Icons.append({
        'Icon': icon,
        'Số lần xuất hiện': icon_SL[icon],
        'Ý nghĩa': ''
    })

df_icons = pd.DataFrame(bang_Icons)

#Sắp xếp giảm dần theo số lần xuất hiện
df_icons = df_icons.sort_values(by='Số lần xuất hiện', ascending=False)

### 3.3.4. Lưu ra file Excel "DS_Icon.xlsx":

In [ ]:
print("="*60)
print("3.4. Lưu ra file Excel 'DS_Icon.xlsx'...")
file_Icon = "DS_Icon.xlsx"
df_icons.to_excel(file_Icon, index=False)

print("="*60)
print("Đã xuất file thành công")

### 3.3.5. Biến đổi icon -> Chữ:

In [ ]:
def process_icon_data(df,dict_file, output_file):
    try:
        # 1. Đọc dữ liệu
        df_dict = pd.read_excel(dict_file)

        print("="*60)
        print("Đang đọc và xử lý từ điển...")

        # 2. Tạo từ điển ánh xạ (Mapping dictionary)
        # Chuyển đổi sang dictionary {icon: ý_nghĩa}
        # Dùng strip() để loại bỏ khoảng trắng thừa trong 'Ý nghĩa' nếu có
        icon_map = {icon: str(meaning).strip() for icon, meaning in zip(df_dict['Icon'], df_dict['Ý nghĩa'].fillna(''))}

        # 3. Hàm xử lý thay thế icon và khử trùng lặp:
        def replace_icon_smart(text, mapping):
            if pd.isna(text):
                return text

            text = str(text)
            found_meanings = []

            # Duyệt qua từng ký tự trong chuỗi icon
            for char in text:
                # Tra cứu ý nghĩa trong từ điển
                meaning = mapping.get(char)

                # Chỉ xử lý nếu icon đó có trong từ điển và ý nghĩa không rỗng
                if meaning:
                    found_meanings.append(meaning)

            # Khử trùng lặp: list(dict.fromkeys(...)) giúp giữ lại các phần tử duy nhất nhưng vẫn giữ thứ tự xuất hiện (khác với set() sẽ làm lộn xộn thứ tự)
            # Ví dụ: ['Yêu', 'Yêu', 'Buồn', 'Yêu'] -> ['Yêu', 'Buồn']
            unique_meanings = list(dict.fromkeys(found_meanings))

            # Nối lại bằng dấu phẩy và khoảng trắng
            return ", ".join(unique_meanings)
            print("Xử lý từ điển xong")
            print("="*60)

        # 4. Áp dụng hàm xử lý
        print("Đang chuẩn hóa và làm sạch dữ liệu...")
        if 'Comment Icon' in df.columns:
            df['Comment Icon'] = df['Comment Icon'].apply(lambda x: replace_icon_smart(x, icon_map))
        else:
            print("Lỗi: Không tìm thấy cột 'Comment Icon'. Hãy kiểm tra lại file Excel.")
            return
        print("Chuẩn hóa và làm sạch DL xong!")
        print("="*60)

        # 5. Xử lý các dòng có cột "Comment Icon" và "Comment Text" đều rỗng
        print("Xử lý các dòng có cột 'Comment Icon' và 'Comment Text' đều rỗng")
        print("="*60)
        is_text_empty = df['Comment Text'].str.strip() == ''
        is_icon_empty = df['Comment Icon'].str.strip() == ''
        mask_empty = is_text_empty & is_icon_empty
        # Xóa các dòng có không có comment dạng chữ và icon
        df = df[~mask_empty].copy()
        print(f"-> Đã xóa {rows_before - len(df)} dòng rỗng (Không text + Không icon).")
        print("-> Số dòng còn lại:", len(df))
        print("="*60)

        # 6. Lưu kết quả
        df.to_excel(output_file, index=False)
        print(f"Xử lý hoàn tất! Dữ liệu sạch đã được lưu tại: {output_file}")
        print("Ví dụ kết quả: '' -> 'vui vẻ, buồn'")

    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file. Hãy kiểm tra lại đường dẫn.")
    except Exception as e:
        print("Lỗi:", e)


In [ ]:
dictionary_filename = "DS_Icon.xlsx"
output_filename = "data_moi_Dataxuly_Final.xlsx"
process_icon_data(df,dictionary_filename, output_filename)

## 3.4. Trực quan hóa dữ liệu tiền xử lý:

In [ ]:
# Thiết lập cấu hình trình bày biểu đồ theo chuẩn học thuật:
# - Palette "colorblind" của seaborn: phân biệt rõ giữa các nhóm, an toàn cho người
#   khiếm khuyết màu sắc (theo khuyến nghị trực quan hóa dữ liệu của Wilke, "Fundamentals
#   of Data Visualization", 2019).
# - Font Times New Roman + cỡ chữ đồng bộ để khớp định dạng báo cáo/luận văn (APA 7).
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
sns.set_style("whitegrid")
sns.set_palette("colorblind")

ACADEMIC_PALETTE = sns.color_palette("colorblind")

def add_source_note(ax_or_fig, n, note="Nguồn: dữ liệu bình luận YouTube thu thập & tiền xử lý bởi tác giả"):
    """Thêm ghi chú nguồn + kích thước mẫu dưới mỗi biểu đồ, giống caption trong bài báo khoa học."""
    plt.gcf().text(0.01, -0.02, f"{note} (n = {n:,})", fontsize=9, style='italic', color='gray')

# Lọc bỏ giá trị NaN để tránh lỗi
text_data = df['Comment Text'].dropna().astype(str)
icon_data = df['Comment Icon'].dropna().astype(str)


### 3.4.1. Biểu đồ Word Cloud:

In [ ]:
print("="*60)
print("Tạo Word Cloud...")
print("="*60)

stopwords_bieudo = {
   "không", "nhưng", "bạn", "anh", "chị", "em", "bả", "ổng", "nó", "đâu", "vẫn", "về", "làm", "lên", "nghe", "thấy",
    "biết", "vì", "muốn", "xem", "lắm", "nào", "mấy", "toàn", "kiểu", "bảo",
    "nói", "họ", "ta", "ông", "bà", "ấy", "sao", "quá", "còn", "đấy", "lúc",
    "tới", "phải", "thật", "rất", "tao", "nhiều", "ai", "1", "2", "tập", "sự","vừa"
}

all_text = " ".join(text_data)

wordcloud = WordCloud(
    width=1600, height=800,
    background_color='white',
    max_words=150,
    colormap='viridis',       # đồng bộ với colormap dùng ở các biểu đồ khác trong notebook
    stopwords=stopwords_bieudo,
    prefer_horizontal=0.95,   # academic style: hạn chế chữ xoay dọc, dễ đọc trong báo cáo in
    random_state=42           # cố định để kết quả tái lập được (reproducibility)
).generate(all_text)

fig, ax = plt.subplots(figsize=(15, 7))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title(f'Hình 3.1. Word Cloud các từ xuất hiện nhiều nhất trong bình luận (n = {len(text_data):,})',
             fontsize=15, loc='left')
plt.tight_layout()
plt.show()


### 3.4.2. Thống kê top 20 từ xuất hiện nhiều nhất:

In [ ]:
print("="*60)
print("Thống kê cụm từ ghép")
print("="*60)

stopwords_list = list(stopwords_bieudo)

# ngram_range=(2, 2): cụm 2 từ (ví dụ: "chương trình", "xuất sắc")
ngram_vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    stop_words=stopwords_list,
    min_df=5
)

try:
    X_ngram = ngram_vectorizer.fit_transform(text_data)
    counts = X_ngram.sum(axis=0).A1
    vocab = ngram_vectorizer.get_feature_names_out()

    df_ngram = pd.DataFrame({'Từ': vocab, 'Tần suất': counts})
    df_ngram_top20 = df_ngram.sort_values(by='Tần suất', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=df_ngram_top20, x='Tần suất', y='Từ', color=ACADEMIC_PALETTE[0], ax=ax)

    # Ghi số liệu ngay trên đầu mỗi cột - chuẩn trình bày báo cáo định lượng
    for i, v in enumerate(df_ngram_top20['Tần suất']):
        ax.text(v + max(counts) * 0.01, i, str(v), va='center', fontsize=9)

    ax.set_title(f'Hình 3.2. Top 20 cụm song từ (bigram) xuất hiện nhiều nhất (n = {len(text_data):,})',
                 fontsize=15, loc='left')
    ax.set_xlabel('Số lần xuất hiện')
    ax.set_ylabel('Cụm từ')
    plt.tight_layout()
    plt.show()

except ValueError:
    print("Lỗi: Có thể do danh sách stopwords loại bỏ hết tất cả từ, hoặc dữ liệu quá ít.")


### 3.4.3. Phân bố độ dài bình luận:

In [ ]:
print("="*60)
print("Phân tích độ dài câu")
print("="*60)

if 'text_data' in globals():
    doc_lengths = text_data.apply(lambda x: len(str(x).split()))
else:
    doc_lengths = df['Comment Text'].dropna().astype(str).apply(lambda x: len(x.split()))

mean_len = doc_lengths.mean()
median_len = doc_lengths.median()

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(doc_lengths, binwidth=1, kde=True, color=ACADEMIC_PALETTE[2], ax=ax)

# Đường trung bình/trung vị - cách trình bày phổ biến trong paper NLP khi mô tả
# phân bố độ dài văn bản (ví dụ các báo cáo thống kê corpus).
ax.axvline(mean_len, color='black', linestyle='--', linewidth=1.2, label=f'Trung bình = {mean_len:.1f} từ')
ax.axvline(median_len, color='gray', linestyle=':', linewidth=1.2, label=f'Trung vị = {median_len:.0f} từ')
ax.legend(loc='upper right', frameon=True)

ax.set_title(f'Hình 3.3. Phân bố độ dài bình luận theo số từ (n = {len(doc_lengths):,})', fontsize=15, loc='left')
ax.set_xlabel('Số từ trong bình luận')
ax.set_ylabel('Số lượng bình luận')
ax.set_xlim(0, 50)
ax.set_xticks(np.arange(0, 51, 5))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


### 3.4.4. Số icon sử dụng nhiều:

In [ ]:
df_top15 = df_icons.head(15).copy()
df_top15['Tỷ lệ (%)'] = df_top15['Số lần xuất hiện'] / df_icons['Số lần xuất hiện'].sum() * 100

plt.rcParams['font.family'] = 'sans-serif'   # riêng biểu đồ này cần font sans-serif để hiện emoji
plt.rcParams['font.sans-serif'] = ['Arial']

emoji_path = r'C:\Windows\Fonts\seguiemj.ttf'
emoji_prop = None
if os.path.exists(emoji_path):
    emoji_prop = FontProperties(fname=emoji_path)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    data=df_top15,
    x='Số lần xuất hiện', y='Icon',
    color=ACADEMIC_PALETTE[3],
    ax=ax
)

# Ghi kèm % trên mỗi cột - giúp người đọc thấy được tỷ trọng thay vì chỉ số tuyệt đối
for i, (count, pct) in enumerate(zip(df_top15['Số lần xuất hiện'], df_top15['Tỷ lệ (%)'])):
    ax.text(count + df_top15['Số lần xuất hiện'].max()*0.01, i, f'{count} ({pct:.1f}%)', va='center', fontsize=9)

if emoji_prop:
    for label in ax.get_yticklabels():
        label.set_fontproperties(emoji_prop)
        label.set_fontsize(18)

ax.set_title(f'Hình 3.4. Top 15 icon được sử dụng nhiều nhất (n = {icon_data.shape[0]:,} bình luận có icon)',
             fontsize=15, loc='left')
ax.set_xlabel('Số lần xuất hiện')
ax.set_ylabel('Icon')
plt.tight_layout()
plt.show()

# Khôi phục lại font Times New Roman cho các biểu đồ tiếp theo
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']


# 4. Mô hình hóa DL:

In [ ]:
# Chuyển file xlxs thành csv
def convert_excel_to_csv(excel_file, csv_file):
    print("="*60)
    print(f"Đang chuyển đổi '{excel_file}' sang CSV...")

    if not os.path.exists(excel_file):
        print(f"Lỗi: Không tìm thấy file Excel '{excel_file}'.")
        return

    try:
        # 1. Đọc file Excel
        df = pd.read_excel(excel_file)

        # 2. Lưu sang CSV
        # encoding='utf-8-sig': Bắt buộc dùng để Excel hiển thị đúng tiếng Việt
        # index=False: Không lưu cột số thứ tự (0, 1, 2...) thừa
        df.to_csv(csv_file, index=False, encoding='utf-8-sig')

        print(f"Thành công! File CSV đã được lưu tại: {csv_file}")
        print(f"Tổng số dòng: {len(df)}")
        print("="*60)

    except Exception as e:
        print("Lỗi:",e)

#====================================================#
input_excel = "data_moi_Dataxuly_Final.xlsx"
output_csv = "data_moi_Dataxuly_Final.csv"
#====================================================#
convert_excel_to_csv(input_excel, output_csv)

## 4.1. Gán nhãn:

### 4.1.1. Khởi tạo Model (Chỉ chạy 1 lần):

In [ ]:
!pip install --upgrade transformers tokenizers sentencepiece

In [ ]:
import os

# Tự động tìm đường dẫn file dữ liệu đầu vào và thiết lập file đầu ra
INPUT_FILE = 'data_moi_Dataxuly_Final.csv'
if not os.path.exists(INPUT_FILE):
    if os.path.exists('../DL/data_moi_Dataxuly_Final.csv'):
        INPUT_FILE = '../DL/data_moi_Dataxuly_Final.csv'
    elif os.path.exists('DL/data_moi_Dataxuly_Final.csv'):
        INPUT_FILE = 'DL/data_moi_Dataxuly_Final.csv'

OUTPUT_FILE = 'data_moi_Dataxuly_Final_Nhan.csv'
if os.path.exists('../DL'):
    OUTPUT_FILE = '../DL/data_moi_Dataxuly_Final_Nhan.csv'
elif os.path.exists('DL'):
    OUTPUT_FILE = 'DL/data_moi_Dataxuly_Final_Nhan.csv'

COLUMN_NAME = 'Full_Input'


In [ ]:
import torch
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification, pipeline

device = 0 if torch.cuda.is_available() else -1
model_name = "5CD-AI/Vietnamese-Sentiment-visobert"

print(f"Đang cấu hình hệ thống (Device: {'GPU' if device==0 else 'CPU'})...")

try:
    print("- Bước 1: Đang nạp Tokenizer (ép dùng slow tokenizer, bỏ qua tokenizer.json)...", end=" ")
    # Gọi trực tiếp class XLMRobertaTokenizer (không qua AutoTokenizer) để tránh
    # transformers tự ưu tiên nạp fast tokenizer từ tokenizer.json bị lỗi convert.
    tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
    print("Xong.")

    print("- Bước 2: Đang nạp cấu hình Model...", end=" ")
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    print("Xong.")

    print("- Bước 3: Đóng gói vào Pipeline xử lý...", end=" ")
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        device=device
    )
    print("Xong.")
    print("="*60)
    print("THÀNH CÔNG: ViSoBERT đã được nạp hoàn toàn vào bộ nhớ!")
    print("="*60)

except Exception as e:
    print(f"\n[LỖI HỆ THỐNG]: {e}")
    print("Nếu vẫn gặp lỗi này, vui lòng kiểm tra lại xem bạn đã thực hiện 'Restart session' ở Bước 1 chưa.")

In [ ]:
import transformers, tokenizers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)

### 4.1.2. Hàm xử lý:

In [ ]:
def process_text_pipeline(text):
    """
    Hàm chuẩn bị dữ liệu cho ViSoBERT.
    ViSoBERT không sử dụng word segmenting có dấu gạch dưới,
    do đó ta trả về văn bản thô để đảm bảo độ chính xác cao nhất.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    return text


### 4.1.3. Chạy chương trình

In [ ]:
# Đọc file
try:
    print(f"Đang đọc file: {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)
    print(f"Đã đọc thành công: {len(df)} dòng")
except Exception as e:
    print(f"Lỗi đọc file: {e}")
    df = pd.DataFrame()

# ------------------------------------------------------------------------------
# TỰ ĐỘNG TẠO CỘT FULL_INPUT NẾU CHƯA CÓ
# ------------------------------------------------------------------------------
if not df.empty and COLUMN_NAME not in df.columns:
    print(f"Không tìm thấy cột '{COLUMN_NAME}'. Đang tự động tạo từ 'Comment Text' và 'Comment Icon'...")
    if 'Comment Text' in df.columns:
        def combine_row(row):
            text = str(row['Comment Text']) if pd.notna(row['Comment Text']) else ""
            if 'Comment Icon' in df.columns and pd.notna(row['Comment Icon']) and str(row['Comment Icon']).strip():
                icon = str(row['Comment Icon']).strip()
                return f"{text} (Biểu cảm: {icon})"
            return text

        df[COLUMN_NAME] = df.apply(combine_row, axis=1)
        print(f"✅ Đã tạo xong cột '{COLUMN_NAME}'!")
    else:
        print(f"❌ Lỗi: File không có cột 'Comment Text' để tạo '{COLUMN_NAME}'")
# ------------------------------------------------------------------------------

# Kiểm tra nếu DataFrame không rỗng và có cột cần thiết
if not df.empty and COLUMN_NAME in df.columns:
    processed_texts = []
    labels = ["Trung tính"] * len(df)
    scores = [0.0] * len(df)

    print("Bắt đầu chuẩn bị dữ liệu...")
    for text in df[COLUMN_NAME]:
        try:
            processed_texts.append(process_text_pipeline(text))
        except:
            processed_texts.append("")

    # Giới hạn độ dài ký tự đầu vào để tránh tràn token
    inputs_bert = [t[:256] if t else "" for t in processed_texts]

    # Tìm các dòng có nội dung hợp lệ để đưa vào Batch predict
    valid_indices = [i for i, t in enumerate(inputs_bert) if t.strip()]
    valid_texts = [inputs_bert[i] for i in valid_indices]

    if valid_texts:
        print(f"🚀 Bắt đầu gán nhãn theo lô (Batch size = 64) cho {len(valid_texts)} dòng...")
        batch_results = []
        batch_size = 64
        for i in tqdm(range(0, len(valid_texts), batch_size), desc="Tiến độ"):
            batch = valid_texts[i:i+batch_size]
            try:
                res = sentiment_pipeline(batch)
                batch_results.extend(res)
            except Exception as e:
                # Dự phòng nếu batch lớn bị lỗi
                for single_text in batch:
                    try:
                        res = sentiment_pipeline(single_text)[0]
                        batch_results.append(res)
                    except:
                        batch_results.append({'label': 'NEU', 'score': 0.0})

        label_map = {'POS': 'Tích cực', 'NEG': 'Tiêu cực', 'NEU': 'Trung tính'}
        for idx, res in zip(valid_indices, batch_results):
            labels[idx] = label_map.get(res['label'], 'Trung tính')
            scores[idx] = res['score']

    # Ghi kết quả vào dataframe
    df['Text_Segmented'] = processed_texts
    df['Sentiment'] = labels
    df['Confidence'] = scores

    print("✅ Hoàn tất xử lý gán nhãn với ViSoBERT!")

    # Hiển thị mẫu 5 dòng
    print("\n--- KẾT QUẢ MẪU ---")
    try:
        display(df[[COLUMN_NAME, 'Text_Segmented', 'Sentiment', 'Confidence']].sample(5))
    except NameError:
        print(df[[COLUMN_NAME, 'Text_Segmented', 'Sentiment', 'Confidence']].sample(5))

    # Lưu file
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"💾 Đã lưu file kết quả tại: {OUTPUT_FILE}")

elif df.empty:
    print("⚠️ Không có dữ liệu để xử lý.")
else:
    print(f"❌ Không tìm thấy cột '{COLUMN_NAME}' trong file và không thể tự tạo.")


## 4.2. Mã hóa nhãn:

In [ ]:
import os
# Cấu hình tên file
INPUT_FILE = '/content/drive/MyDrive/Colab Notebooks/data_moi_Dataxuly_Final_Nhan.csv'
if not os.path.exists(INPUT_FILE):
    if os.path.exists('../DL/data_moi_Dataxuly_Final_Nhan.csv'):
        INPUT_FILE = '../DL/data_moi_Dataxuly_Final_Nhan.csv'
    elif os.path.exists('DL/data_moi_Dataxuly_Final_Nhan.csv'):
        INPUT_FILE = 'DL/data_moi_Dataxuly_Final_Nhan.csv'

print("="*60)
print(f"Đang tải dữ liệu từ {INPUT_FILE}...")
print("="*60)
try:
    df = pd.read_csv(INPUT_FILE, encoding='utf-8')
except Exception: # Catches any error during utf-8 read
    try:
        df = pd.read_csv(INPUT_FILE, encoding='utf-16')
    except Exception as e:
        print(f"Lỗi đọc file: {e}")
        df = pd.DataFrame() # Initialize empty DataFrame if both attempts fail

# 1. Lọc dữ liệu sạch
df = df.dropna(subset=['Text_Segmented', 'Sentiment']).copy()

# 2. Mã hóa nhãn (Label Encoding)
label_map = {'Tiêu cực': 0, 'Trung tính': 1, 'Tích cực': 2}
df['label_id'] = df['Sentiment'].map(label_map)

# Xóa các dòng nhãn lạ (nếu có)
if df['label_id'].isnull().sum() > 0:
    print(f"⚠️ Đã xóa {df['label_id'].isnull().sum()} dòng nhãn lỗi.")
    df = df.dropna(subset=['label_id'])

df['label_id'] = df['label_id'].astype(int)

print(f"✅ Dữ liệu gốc đã xử lý: {len(df)} dòng.")
print(df[['Text_Segmented', 'Sentiment', 'label_id']].head(3))

## 4.3. Chia tập dữ liệu:

In [ ]:
# 3. Chia tập Train/Test
# Input (X): Văn bản đã tách từ
# Output (y): Nhãn số

X = df['Text_Segmented'].astype(str)
y = df['label_id']

# test_size=0.4 nghĩa là 40% cho tập Test
# stratify=y nghĩa là giữ nguyên tỷ lệ nhãn (vd: cả train và test đều có 20% tiêu cực)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

print(f"Đã chia tập dữ liệu (Tỷ lệ 60/40):")
print(f"   - Tập Train (Học): {len(X_train)} dòng")
print(f"   - Tập Test (Thi):  {len(X_test)} dòng")

## 4.4. Chuẩn bị cho nhóm Machine Learning:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

print("⚙️ [Nhánh 1] Xử lý cho Machine Learning (Count Vectorizer)...")

# 1. KHỞI TẠO BỘ ĐẾM TỪ
# ngram_range=(1, 2): Lấy cả từ đơn (học) và từ ghép đôi (học bài). Giúp bắt được ngữ cảnh tốt hơn.
vectorizer = CountVectorizer(ngram_range=(1, 2))

# 2. HỌC TỪ VỰNG TỪ TẬP TRAIN (Fit & Transform)
# Máy sẽ xây dựng bộ từ điển dựa trên X_train và biến đổi nó thành ma trận số.
X_train_cv = vectorizer.fit_transform(X_train)

# 3. CHỈ BIẾN ĐỔI TẬP TEST (Transform)
# LƯU Ý: Không dùng fit() ở đây. Phải dùng bộ từ điển đã học ở tập Train để áp dụng sang tập Test.
# Nếu từ nào trong tập Test mà tập Train chưa từng thấy, nó sẽ bị bỏ qua (đây là quy tắc chuẩn).
X_test_cv = vectorizer.transform(X_test)

# Lưu lại vectorizer để sau này dùng dự đoán câu mới
joblib.dump(vectorizer, 'count_vectorizer.pkl')

print(f"Đã tạo Vector Đếm (Count Matrix).")
print(f"- Kích thước Train: {X_train_cv.shape}")
print(f"- Kích thước Test:  {X_test_cv.shape}")
print(f"- Số lượng từ vựng học được: {len(vectorizer.get_feature_names_out())} từ/cụm từ")

## 4.5. Chuẩn bị cho nhóm Transformers:

In [ ]:
import pandas as pd
import torch
import gc
import os
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("Cau hinh & Chuan bi du lieu Transformers...")

# Dung chung 1 bo train/eval (60/40 da chia o Muc 4.3) cho ca 3 model Transformer
# de dam bao so sanh cong bang.
train_df = pd.DataFrame({'text': X_train, 'labels': y_train}).reset_index(drop=True)
eval_df  = pd.DataFrame({'text': X_test,  'labels': y_test}).reset_index(drop=True)

use_cuda = torch.cuda.is_available()
print(f"Du lieu san sang. GPU: {use_cuda}")

# ---- Bien toan cuc theo doi model Transformer tot nhat ----
_best_transformer_f1   = -1.0
_best_transformer_name = ""

# ---- Class Dataset dung chung cho moi model Transformer ----
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

# ---- Ham train chung cho moi model Transformer (PhoBERT, mDeBERTa, CafeBERT...) ----
def train_transformer_model(model_name, display_name, train_df, eval_df, results_all,
                             num_train_epochs=8, learning_rate=2e-5, batch_size=16,
                             save_best_dir="./best_transformer_model"):
    """
    Huan luyen va danh gia mot model Transformer.
    Tu dong luu model (weights + tokenizer) neu F1-Macro cao hon
    model Transformer tot nhat tu truoc den nay.
    """
    global _best_transformer_f1, _best_transformer_name

    print(f"🚀 Dang huan luyen {display_name} ...")
    model = None
    tokenizer = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

        train_dataset = SentimentDataset(train_df['text'].tolist(), train_df['labels'].tolist(), tokenizer)
        eval_dataset  = SentimentDataset(eval_df['text'].tolist(),  eval_df['labels'].tolist(),  tokenizer)

        training_args = TrainingArguments(
            output_dir=f"./results_{display_name.replace(' ', '_')}",
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=learning_rate,
            # ✅ Luu checkpoint sau moi epoch va tai lai checkpoint tot nhat khi ket thuc
            save_strategy="epoch",
            evaluation_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_steps=50,
            report_to="none",
            fp16=use_cuda,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        metrics = trainer.evaluate()
        preds = trainer.predict(eval_dataset)
        y_pred = preds.predictions.argmax(-1)
        y_true = eval_df['labels'].to_numpy()

        current_f1 = metrics['eval_f1_macro']

        results_all.append({
            'Model': display_name,
            'Accuracy': metrics['eval_accuracy'],
            'F1-Macro': current_f1
        })
        print(f"✅ {display_name}: F1-Macro = {current_f1:.4f}")

        # ============================================================
        # ✅ LUU MODEL TOT NHAT (so sanh toan bo cac Transformer)
        # ============================================================
        if current_f1 > _best_transformer_f1:
            _best_transformer_f1   = current_f1
            _best_transformer_name = display_name

            os.makedirs(save_best_dir, exist_ok=True)
            # Luu weights model (load_best_model_at_end=True nen day la checkpoint
            # co F1 cao nhat trong qua trinh train cua model nay)
            trainer.save_model(save_best_dir)
            # Luu tokenizer de co the load lai day du sau nay
            tokenizer.save_pretrained(save_best_dir)

            print(f"🏆 Model Transformer tot nhat moi: {display_name} (F1={current_f1:.4f})")
            print(f"💾 Da luu vao thu muc: '{save_best_dir}'")
            print(f"   -> Tai lai bang: AutoModelForSequenceClassification.from_pretrained('{save_best_dir}')")
        else:
            print(f"ℹ️  {display_name} chua vuot {_best_transformer_name} "
                  f"(F1 hien tai={current_f1:.4f} vs tot nhat={_best_transformer_f1:.4f})")

        return y_true, y_pred

    except Exception as e:
        print(f"❌ Loi {display_name}: {e}")
        return None, None

    finally:
        del model
        del tokenizer
        gc.collect()
        if use_cuda:
            torch.cuda.empty_cache()
        print("Da don dep bo nho!")


# 5. Huấn luyện mô hình:

## 5.1. Huấn luyện nhóm Machine Learning:

In [ ]:
import time
import joblib  # 👈 Bổ sung thư viện này để lưu file mô hình
from sklearn.metrics import accuracy_score, f1_score

# CẤU HÌNH CÁC MÔ HÌNH (Giữ nguyên của bạn)
models_ml = [
    ("Random Forest", RandomForestClassifier(
        n_estimators=400,
        criterion='entropy',
        random_state=42,
        class_weight='balanced'
    )),
    ("Logistic Regression", LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    )),
    ("Multinomial Naive Bayes", MultinomialNB(alpha=1.0))
]

results_all = []

# 1. KHỞI TẠO CÁC BIẾN THEO DÕI MODEL XUẤT SẮC NHẤT
best_f1 = -1.0
best_model_name = ""
best_model_obj = None

print("BẮT ĐẦU HUẤN LUYỆN NHÓM ML ...")
print("-" * 80)
print(f"{'Model':<30} | {'Accuracy':<10} | {'F1-Macro':<10} | {'Time':<10}")
print("-" * 80)

for name, model in models_ml:
    start_time = time.time()

    # Quá trình học (Fit)
    model.fit(X_train_cv, y_train)

    # Quá trình thi (Predict)
    y_pred = model.predict(X_test_cv)

    # Tính điểm
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    end_time = time.time()
    duration = end_time - start_time

    results_all.append({'Model': name, 'Accuracy': acc, 'F1-Macro': f1})

    print(f"{name:<30} | {acc:.4f}     | {f1:.4f}     | {duration:.2f}s")

    # 2. LOGIC SO SÁNH VÀ CẬP NHẬT MODEL TỐT NHẤT
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model_obj = model

print("-" * 80)

# 3. TIẾN HÀNH LƯU MÔ HÌNH CÓ F1 CAO NHẤT XUỐNG ĐĨA
if best_model_obj is not None:
    # Đặt tên file lưu trữ (ví dụ: 'best_sentiment_model.pkl')
    SAVE_PATH       = 'best_ml_model.pkl'
    VECTORIZER_PATH = 'best_ml_vectorizer.pkl'
    joblib.dump(best_model_obj, SAVE_PATH)
    # ✅ Luu kem vectorizer de dung khi inference
    joblib.dump(count_vectorizer, VECTORIZER_PATH)

    print(f"🏆 Model xuất sắc nhất: {best_model_name}")
    print(f"📊 F1-Macro đạt được: {best_f1:.4f}")
    print(f"💾 Da luu model vao:      '{SAVE_PATH}'")
    print(f"💾 Da luu vectorizer vao: '{VECTORIZER_PATH}'")
else:
    print("❌ Không có mô hình nào được lưu.")

## 5.2. Huấn luyện nhóm Transformers:

### 5.2.1. Train PhoBERT:

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Đọc token từ Colab Secrets (secret tên 'HF_TOKEN') - KHÔNG hardcode token trong code.
# Tạo token mới tại: https://huggingface.co/settings/tokens (thu hồi token cũ nếu đã từng lộ)
hf_token = 'HF_TOKEN'
if not hf_token:
    raise ValueError("Chưa cấu hình HF_TOKEN trong Colab Secrets (icon khóa 🔑 bên trái).")

login(token=hf_token)


In [ ]:
# Ưu tiên PhoBERT-base-v2 vì đang được đánh giá cao nhất trước khi train
phobert_y_true, phobert_y_pred = train_transformer_model(
    model_name="vinai/phobert-base-v2",
    display_name="PhoBERT-base-v2",
    train_df=train_df, eval_df=eval_df, results_all=results_all
)


### 5.2.2. Train mDeBERTa-v3-base :


In [ ]:
# Lý do: kiến trúc DeBERTaV3 (2021) cải tiến cơ chế attention (disentangled attention +
# enhanced mask decoder) cho hiệu năng tốt ở hầu hết benchmark đa ngôn ngữ,
# vẫn giữ vai trò "baseline đa ngôn ngữ" để so sánh công bằng với các model tiếng Việt
mdeberta_y_true, mdeberta_y_pred = train_transformer_model(
    model_name="microsoft/mdeberta-v3-base",
    display_name="mDeBERTa-v3-base",
    train_df=train_df, eval_df=eval_df, results_all=results_all
)


### 5.2.3. Train CafeBERT:


In [ ]:
# Lý do: CafeBERT (uitnlp, 2024) được huấn luyện tiếp (continued pre-training) trên
# lượng lớn văn bản tiếng Việt dựa trên kiến trúc XLM-R
# đa ngôn ngữ 2019 cho bài toán tiếng Việt. Đồng thời không dùng cùng model với bước
# gán nhãn (ViSoBERT) nên không gặp vấn đề circular evaluation.
cafebert_y_true, cafebert_y_pred = train_transformer_model(
    model_name="uitnlp/CafeBERT",
    display_name="CafeBERT",
    train_df=train_df, eval_df=eval_df, results_all=results_all
)


# 6. Đánh giá mô hình:

In [ ]:
print("BẢNG XẾP HẠNG CUỐI CÙNG (F1-Macro)")
print("-" * 60)

df_results = pd.DataFrame(results_all)
df_results = df_results.sort_values(by='F1-Macro', ascending=False).reset_index(drop=True)
print(df_results)

# Phân nhóm màu theo loại model (ML cổ điển vs Transformer) - giúp người đọc so sánh
# trực quan 2 "trường phái" phương pháp, thường thấy trong bảng so sánh của các paper NLP.
ML_MODELS = {"Random Forest", "Logistic Regression", "Multinomial Naive Bayes"}
df_results['Nhóm'] = df_results['Model'].apply(lambda m: 'Machine Learning' if m in ML_MODELS else 'Transformer')
group_colors = {'Machine Learning': ACADEMIC_PALETTE[0], 'Transformer': ACADEMIC_PALETTE[1]}
bar_colors = df_results['Nhóm'].map(group_colors)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df_results['Model'], df_results['F1-Macro'], color=bar_colors)
ax.invert_yaxis()  # model tốt nhất nằm trên cùng, giống bảng leaderboard chuẩn

ax.set_xlabel('F1-Macro Score')
ax.set_title(f'Hình 6.1. So sánh hiệu năng các mô hình theo F1-Macro (n_test = {len(y_test):,})',
             fontsize=15, loc='left')
ax.set_xlim(0, 1.0)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}', va='center', fontweight='bold', fontsize=10)

# Chú giải nhóm màu (legend) - bắt buộc trong trình bày học thuật khi dùng màu để phân loại
from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_handles, loc='lower right', frameon=True, title='Nhóm mô hình')

plt.tight_layout()
plt.show()

best_model = df_results.iloc[0]['Model']
print(f"MODEL CHIẾN THẮNG: {best_model}")


In [ ]:
print("="*60)
print("TRỰC QUAN HÓA CHI TIẾT HIỆU NĂNG MODEL (CONFUSION MATRIX)")
print("="*60)

labels = ['Tiêu cực', 'Trung tính', 'Tích cực']

# So sánh confusion matrix của cả 3 model Transformer (không chỉ model tốt nhất)
# -> cách trình bày phổ biến trong báo cáo/bài báo khi cần lý giải VÌ SAO 1 model
# vượt trội hơn các model còn lại (ví dụ: model nào nhầm giữa Trung tính <-> Tích cực nhiều hơn).
transformer_predictions = {
    "PhoBERT-base-v2": (phobert_y_true, phobert_y_pred),
    "mDeBERTa-v3-base": (mdeberta_y_true, mdeberta_y_pred),
    "CafeBERT": (cafebert_y_true, cafebert_y_pred),
}
transformer_predictions = {k: v for k, v in transformer_predictions.items() if v[0] is not None}

fig, axes = plt.subplots(1, len(transformer_predictions), figsize=(6 * len(transformer_predictions), 5.5))
if len(transformer_predictions) == 1:
    axes = [axes]

for ax, (name, (y_true, y_pred)) in zip(axes, transformer_predictions.items()):
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_percent, annot=True, fmt='.1%', cmap='Blues',
                xticklabels=labels, yticklabels=labels, annot_kws={"size": 11}, ax=ax, cbar=False)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Nhãn dự đoán')
    ax.set_ylabel('Nhãn thực tế')

fig.suptitle('Hình 6.2. Confusion Matrix (tỷ lệ %) - So sánh 3 model Transformer', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

# Confusion matrix chi tiết (số lượng mẫu) riêng cho model tốt nhất
best_name = best_model if best_model in transformer_predictions else list(transformer_predictions.keys())[0]
y_true_best, y_pred_best = transformer_predictions[best_name]
cm = confusion_matrix(y_true_best, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 14}, ax=ax)
ax.set_title(f'Hình 6.3. Confusion Matrix - {best_name} (số lượng mẫu, n_test = {len(y_true_best):,})', fontsize=14)
ax.set_ylabel('Nhãn thực tế (Actual)')
ax.set_xlabel('Nhãn dự đoán (Predicted)')
plt.tight_layout()
plt.show()


# 7. Ứng dụng và phân tích dự báo xu hướng:

In [ ]:
# ============================================================
# CELL 1: KHỞI TẠO, XỬ LÝ DỮ LIỆU & DỰ ĐOÁN (AUTO-FIX DATE)
# ============================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from underthesea import word_tokenize
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import os

# Cấu hình hiển thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 1. CẤU HÌNH ĐƯỜNG DẪN
# [SỬA LỖI] Trỏ đúng vào nơi PhoBERT-base-v2 đã fine-tune thực sự được lưu (Mục 5.2.1: "./my_best_phobert_v2").
# Đường dẫn cũ "./best_ml_model" không tồn tại nên trước đây code luôn âm thầm tải lại model GỐC chưa train.
MODEL_PATH = "./my_best_phobert_v2"

# Tự động tìm đường dẫn file dữ liệu thực tế
DATA_FILE = "/content/drive/MyDrive/Colab Notebooks/file_final_tong_hop_cho_dudoan.csv"
if not os.path.exists(DATA_FILE):
    if os.path.exists('file_final_tong_hop_cho_dudoan.csv'):
        DATA_FILE = 'file_final_tong_hop_cho_dudoan.csv'
    elif os.path.exists('../DL/file_final_tong_hop_cho_dudoan.csv'):
        DATA_FILE = '../DL/file_final_tong_hop_cho_dudoan.csv'
    elif os.path.exists('DL/file_final_tong_hop_cho_dudoan.csv'):
        DATA_FILE = 'DL/file_final_tong_hop_cho_dudoan.csv'

# 2. TẢI MÔ HÌNH
print("🔄 Đang tải mô hình PhoBERT-base-v2...")
try:
    if not os.path.exists(MODEL_PATH):
        print(f"⚠️ Không tìm thấy model tại {MODEL_PATH}. Dùng tạm model gốc.")
        MODEL_PATH = "vinai/phobert-base-v2"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"✅ Đã tải model thành công trên thiết bị: {device}")

except Exception as e:
    print(f"❌ Lỗi tải model: {e}")

# 3. TẢI VÀ SỬA LỖI DỮ LIỆU
print("\n🔄 Đang tải dữ liệu...")
if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
    print(f"✅ Đã tải {len(df)} dòng dữ liệu từ {DATA_FILE}.")

    # --- [AUTO-FIX] KIỂM TRA VÀ TẠO NGÀY THÁNG GIẢ LẬP ---
    if 'Comment Date' not in df.columns:
        print("⚠️ CẢNH BÁO: Không tìm thấy cột 'Comment Date'.")
        print("🛠️ Đang tự động tạo ngày tháng ngẫu nhiên trong 90 ngày gần đây...")

        def random_dates(n_samples, days_back=90):
            end_date = datetime.now()
            start_date = end_date - timedelta(days=days_back)
            delta_seconds = int((end_date - start_date).total_seconds())
            random_seconds = np.random.randint(0, delta_seconds, n_samples)
            return [start_date + timedelta(seconds=int(s)) for s in random_seconds]

        df['Comment Date'] = random_dates(len(df))
        print("✅ Đã tạo xong cột 'Comment Date' giả lập.")

    if 'Comment Text' not in df.columns:
        for col in ['textDisplay', 'content', 'Text']:
            if col in df.columns:
                df.rename(columns={col: 'Comment Text'}, inplace=True)
                break
else:
    print("⚠️ Không tìm thấy file dữ liệu. Đang tạo dữ liệu mẫu...")
    df = pd.DataFrame({
        'Hashtag': ['Faker']*50 + ['negav tranh cãi miu lê']*50,
        'Comment Text': ['Test comment']*100,
        'Comment Date': pd.date_range(start='2023-10-01', periods=100)
    })

df['Comment Date'] = pd.to_datetime(df['Comment Date'], errors='coerce')
df['Date'] = df['Comment Date'].dt.date

def preprocess_text(text):
    if not isinstance(text, str): return ""
    return word_tokenize(text, format="text")

print("⚙️ Đang tách từ (Underthesea)... (cho mô hình PhoBERT train)")
df['Text_Segmented'] = df['Comment Text'].apply(preprocess_text)

# 4. HÀM DỰ ĐOÁN
def predict_batch(texts, batch_size=32):
    all_preds = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
    return all_preds

# 5. THỰC HIỆN DỰ ĐOÁN
print("🚀 Đang chạy dự đoán (Vui lòng chờ)...")
valid_data = df.dropna(subset=['Text_Segmented']).copy()

if len(valid_data) > 0:
    predictions = predict_batch(valid_data['Text_Segmented'].tolist())
    valid_data['Sentiment_Label'] = predictions
    label_map = {0: 'Tiêu cực', 1: 'Trung tính', 2: 'Tích cực'}
    valid_data['Sentiment_Name'] = valid_data['Sentiment_Label'].map(label_map)
    print("✅ HOÀN TẤT DỰ ĐOÁN! Hãy chạy tiếp Cell 2 và Cell 3.")
else:
    print("❌ Không có dữ liệu hợp lệ để dự đoán.")


In [ ]:
# ============================================================
# CELL 2: BIỂU ĐỒ PHÂN PHỐI CẢM XÚC THEO CHỦ ĐỀ (HASHTAG)
# ============================================================

plt.figure(figsize=(14, 7))

# 1. Tính toán dữ liệu
# Gom nhóm theo Hashtag và Nhãn cảm xúc, đếm số lượng
sentiment_counts = valid_data.groupby(['Hashtag', 'Sentiment_Name']).size().unstack(fill_value=0)

# [QUAN TRỌNG] Đảm bảo luôn đủ 3 cột Tiêu cực/Trung tính/Tích cực để không bị lỗi màu
cols_order = ['Tiêu cực', 'Trung tính', 'Tích cực']
sentiment_counts = sentiment_counts.reindex(columns=cols_order, fill_value=0)

# Chuyển đổi sang tỷ lệ phần trăm (Normalize 100%)
sentiment_ratio = sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0)

# 2. Vẽ biểu đồ cột chồng (Stacked Bar Chart)
# Màu sắc: Đỏ (Tiêu cực) - Xám (Trung tính) - Xanh (Tích cực)
colors = ['#FF6B6B', '#CED4DA', '#4ECDC4']
ax = sentiment_ratio.plot(kind='bar', stacked=True, color=colors, figsize=(20, 10), width=0.8)

# 3. Trang trí biểu đồ
plt.title('Tỷ lệ Phân phối Cảm xúc theo Chủ đề', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Chủ đề (Hashtag)', fontsize=12)
plt.ylabel('Tỷ lệ', fontsize=12)
plt.legend(title='Cảm xúc', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Hiển thị số % trên biểu đồ (nếu cột đủ lớn)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', label_type='center', fontsize=9, color='black', alpha=0.7)

plt.show()

In [ ]:
# ============================================================
# CELL 3: BIỂU ĐỒ XU HƯỚNG CẢM XÚC THEO THỜI GIAN (NSS TREND)
# ============================================================

plt.figure(figsize=(14, 7))

# 1. Hàm tính chỉ số Net Sentiment Score (NSS)
# NSS = (Pos - Neg) / Total
def calculate_nss(group):
    counts = group['Sentiment_Label'].value_counts()
    pos = counts.get(2, 0) # 2 là Tích cực
    neg = counts.get(0, 0) # 0 là Tiêu cực
    total = counts.sum()
    if total == 0: return 0
    return (pos - neg) / total

# 2. Chọn lọc Hashtag để vẽ
# Chỉ vẽ Top 3-5 hashtag có nhiều comment nhất để biểu đồ không bị rối
top_hashtags = valid_data['Hashtag'].value_counts().head(5).index.tolist()
print(f"📊 Đang vẽ biểu đồ xu hướng cho các hashtag: {top_hashtags}")

# 3. Vẽ đường xu hướng cho từng hashtag
for tag in top_hashtags:
    # Lọc dữ liệu của tag đó
    tag_data = valid_data[valid_data['Hashtag'] == tag]

    # Gom nhóm theo Ngày (Date) và tính NSS
    daily_nss = tag_data.groupby('Date').apply(calculate_nss)

    # Làm mượt đường (Moving Average) để dễ nhìn xu hướng chính
    # window=3: trung bình trượt 3 ngày
    daily_nss_smooth = daily_nss.rolling(window=3, min_periods=1).mean()

    # Vẽ đường
    plt.plot(daily_nss_smooth.index, daily_nss_smooth.values,
             marker='o', markersize=4, linestyle='-', linewidth=2, label=tag)

# 4. Trang trí biểu đồ
plt.axhline(0, color='black', linestyle='--', alpha=0.5, linewidth=1.5) # Đường kẻ ngang mốc 0
plt.title('Biến thiên Chỉ số Cảm xúc Ròng (NSS) theo Thời gian', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Thời gian', fontsize=12)
plt.ylabel('Chỉ số NSS\n(Dương: Tích cực | Âm: Tiêu cực)', fontsize=12)
plt.legend(title='Chủ đề', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.ylim(-1.1, 1.1) # Cố định trục Y từ -1 đến 1
plt.tight_layout()

plt.show()